In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/akashgoyalll/hindi-english-truncated-corpus/Hindi_English_Truncated_Corpus.csv


In [13]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.callbacks import EarlyStopping
import pickle

# ==========================================
# 1. LOAD AND CLEAN THE EXACT DATASET
# ==========================================
data_path = '/kaggle/input/datasets/akashgoyalll/hindi-english-truncated-corpus/Hindi_English_Truncated_Corpus.csv' 
df = pd.read_csv(data_path)
df.dropna(inplace=True)

# Take 30,000 sentences
lines = df.iloc[:30000].copy()

# Add start and end tokens
lines['hindi_sentence'] = lines['hindi_sentence'].apply(lambda x: '<start> ' + str(x) + ' <end>')

# ==========================================
# 2. TOKENIZATION & PADDING (MEMORY FIX APPLIED HERE)
# ==========================================
# FIX 1: Limit vocabulary to top 10,000 words to save memory
MAX_VOCAB_SIZE = 10000
# FIX 2: Limit sentence length to 30 words to save memory
MAX_SEQ_LEN = 30

# English
eng_tokenizer = Tokenizer(filters='', num_words=MAX_VOCAB_SIZE, oov_token='<unk>')
eng_tokenizer.fit_on_texts(lines['english_sentence'])
eng_vocab_size = min(len(eng_tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

# Hindi
hin_tokenizer = Tokenizer(filters='', num_words=MAX_VOCAB_SIZE, oov_token='<unk>')
hin_tokenizer.fit_on_texts(lines['hindi_sentence'])
hin_vocab_size = min(len(hin_tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

# Convert to sequences and pad (using the maxlen limit)
encoder_input_data = pad_sequences(eng_tokenizer.texts_to_sequences(lines['english_sentence']), maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
decoder_input_data = pad_sequences(hin_tokenizer.texts_to_sequences(lines['hindi_sentence']), maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

# Target data (shifted by 1 timestep)
decoder_target_data = np.zeros_like(decoder_input_data)
decoder_target_data[:, 0:-1] = decoder_input_data[:, 1:]

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
# FIX 3: Slightly reduce latent dimension to save memory
latent_dim = 128 
embedding_dim = 100

# Encoder
encoder_inputs = Input(shape=(MAX_SEQ_LEN,), name='encoder_inputs')
enc_emb = Embedding(eng_vocab_size, embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(MAX_SEQ_LEN,), name='decoder_inputs')
dec_emb_layer = Embedding(hin_vocab_size, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(hin_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Compile
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# ==========================================
# 4. TRAINING
# ==========================================
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Starting training with memory optimizations...")
model.fit(
    [encoder_input_data, decoder_input_data], 
    decoder_target_data,
    batch_size=64,
    epochs=50, 
    validation_split=0.2,
    callbacks=[early_stop]
)

# ==========================================
# 5. SAVE FILES FOR FRONTEND
# ==========================================
model.save('eng_hin_translation_model.h5')

frontend_data = {
    'eng_tokenizer': eng_tokenizer,
    'hin_tokenizer': hin_tokenizer,
    'max_eng_len': MAX_SEQ_LEN,
    'max_hin_len': MAX_SEQ_LEN,
    'latent_dim': latent_dim
}

with open('tokenizer_data.pkl', 'wb') as f:
    pickle.dump(frontend_data, f)
    
print("\nSuccess! Download 'eng_hin_translation_model.h5' and 'tokenizer_data.pkl'.")

Starting training with memory optimizations...
Epoch 1/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 22s 47ms/step - accuracy: 0.4730 - loss: 4.8012 - val_accuracy: 0.5232 - val_loss: 3.3809
Epoch 2/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5241 - loss: 3.3865 - val_accuracy: 0.5369 - val_loss: 3.2701
Epoch 3/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5381 - loss: 3.2540 - val_accuracy: 0.5414 - val_loss: 3.2064
Epoch 4/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5402 - loss: 3.2224 - val_accuracy: 0.5522 - val_loss: 3.1549
Epoch 5/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5497 - loss: 3.1493 - val_accuracy: 0.5577 - val_loss: 3.1076
Epoch 6/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5505 - loss: 3.1346 - val_accuracy: 0.5596 - val_loss: 3.0635
Epoch 7/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5585 - loss: 3.0599 - val_accuracy: 0.5645 - val_loss: 3.0290
Epoch 8/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s


Success! Download 'eng_hin_translation_model.h5' and 'tokenizer_data.pkl'.


In [18]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.callbacks import EarlyStopping
import pickle

# ==========================================
# 1. LOAD AND CLEAN DATA
# ==========================================
data_path = '/kaggle/input/datasets/akashgoyalll/hindi-english-truncated-corpus/Hindi_English_Truncated_Corpus.csv' 
df = pd.read_csv(data_path)
df.dropna(inplace=True)

# Using 30,000 sentences 
lines = df.iloc[:30000].copy()
lines['hindi_sentence'] = lines['hindi_sentence'].apply(lambda x: '<start> ' + str(x) + ' <end>')

# ==========================================
# 2. TOKENIZATION & PADDING (MEMORY SAFE)
# ==========================================
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 30

# English
eng_tokenizer = Tokenizer(filters='', num_words=MAX_VOCAB_SIZE, oov_token='<unk>')
eng_tokenizer.fit_on_texts(lines['english_sentence'])
eng_vocab_size = min(len(eng_tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

# Hindi
hin_tokenizer = Tokenizer(filters='', num_words=MAX_VOCAB_SIZE, oov_token='<unk>')
hin_tokenizer.fit_on_texts(lines['hindi_sentence'])
hin_vocab_size = min(len(hin_tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

# Convert to sequences
encoder_input_data = pad_sequences(eng_tokenizer.texts_to_sequences(lines['english_sentence']), maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
decoder_input_data = pad_sequences(hin_tokenizer.texts_to_sequences(lines['hindi_sentence']), maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

# Target data (shifted by 1 timestep)
decoder_target_data = np.zeros_like(decoder_input_data)
decoder_target_data[:, 0:-1] = decoder_input_data[:, 1:]

# BUG FIX: Explicitly cast to int32 numpy arrays to prevent the "Eager Execution" crash
encoder_input_data = np.array(encoder_input_data, dtype=np.int32)
decoder_input_data = np.array(decoder_input_data, dtype=np.int32)
decoder_target_data = np.array(decoder_target_data, dtype=np.int32)

# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
latent_dim = 128 
embedding_dim = 100

# Encoder
encoder_inputs = Input(shape=(MAX_SEQ_LEN,), name='encoder_inputs')
enc_emb = Embedding(eng_vocab_size, embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(MAX_SEQ_LEN,), name='decoder_inputs')
dec_emb_layer = Embedding(hin_vocab_size, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(hin_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Compile
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# ==========================================
# 4. TRAINING (SET TO 100 EPOCHS)
# ==========================================
# Patience set to 10 so it trains longer before giving up
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("Starting clean training run for up to 100 epochs...")
model.fit(
    x=[encoder_input_data, decoder_input_data], 
    y=decoder_target_data,
    batch_size=64,
    epochs=100, 
    validation_split=0.2,
    callbacks=[early_stop]
)

# ==========================================
# 5. SAVE FILES 
# ==========================================
# BUG FIX: Saving as .keras instead of .h5 prevents graph corruption in newer TF versions
model.save('eng_hin_translation_model_final.keras')

frontend_data = {
    'eng_tokenizer': eng_tokenizer,
    'hin_tokenizer': hin_tokenizer,
    'max_eng_len': MAX_SEQ_LEN,
    'max_hin_len': MAX_SEQ_LEN,
    'latent_dim': latent_dim
}

with open('tokenizer_data.pkl', 'wb') as f:
    pickle.dump(frontend_data, f)
    
print("\nSuccess! Download 'eng_hin_translation_model_final.keras' and 'tokenizer_data.pkl'.")

Starting clean training run for up to 100 epochs...
Epoch 1/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 20s 47ms/step - accuracy: 0.4684 - loss: 4.7801 - val_accuracy: 0.5282 - val_loss: 3.3862
Epoch 2/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5217 - loss: 3.4137 - val_accuracy: 0.5337 - val_loss: 3.2962
Epoch 3/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5297 - loss: 3.3113 - val_accuracy: 0.5420 - val_loss: 3.2170
Epoch 4/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5340 - loss: 3.2619 - val_accuracy: 0.5482 - val_loss: 3.1662
Epoch 5/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5458 - loss: 3.1767 - val_accuracy: 0.5568 - val_loss: 3.1140
Epoch 6/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5523 - loss: 3.1224 - val_accuracy: 0.5580 - val_loss: 3.0808
Epoch 7/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 46ms/step - accuracy: 0.5551 - loss: 3.0844 - val_accuracy: 0.5615 - val_loss: 3.0458
Epoch 8/100
375/375 ━━━━━━━━━━━